In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("SilverToGold").getOrCreate()

# Dynamically find the project root regardless of who runs it
NOTEBOOK_DIR = os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..', '..'))

DATA_DIR = os.path.join(PROJECT_ROOT, "data")
SILVER_DIR = os.path.join(DATA_DIR, "silver")
GOLD_DIR = os.path.join(DATA_DIR, "gold")

# ------------------------------------------------------------------------------
# Load Base Dimension Tables for Joining
# ------------------------------------------------------------------------------
players_ids = spark.read.parquet(os.path.join(SILVER_DIR, 'ids', 'players_ids')).select(F.col("player").alias("join_player"), F.col("player_id"))
teams_ids = spark.read.parquet(os.path.join(SILVER_DIR, 'ids', 'teams_ids')).select(F.col("team").alias("join_team"), F.col("team_id"))


In [ ]:
# ==============================================================================
# Dataset : dim_players
# Layer   : Silver -> Gold
# Purpose : Build Star Schema (Join with Dimensions)
# ==============================================================================

print("=" * 80)
print("Processing Gold Table : dim_players")
print("=" * 80)


In [ ]:
# ------------------------------------------------------------------------------
# Step 1 : Read Silver Dataset
# ------------------------------------------------------------------------------
dataset = "players_ids"
gold_name = "dim_players"
folder = "ids"

df = spark.read.parquet(os.path.join(SILVER_DIR, folder, dataset))
rows_before = df.count()
print(f"Silver Rows Read : {rows_before}")
df.show(5)


In [ ]:
# ------------------------------------------------------------------------------
# Step 3 : Write Gold Dataset
# ------------------------------------------------------------------------------
df.coalesce(1).write.mode("overwrite").csv(os.path.join(GOLD_DIR, gold_name), header=True)


In [ ]:
# ------------------------------------------------------------------------------
# Step 4 : ETL Summary
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("ETL EXECUTION SUMMARY")
print("=" * 80)
print(f"Gold Table           : dim_players")
print(f"Rows Written         : {df.count()}")
print(f"Gold Location        : {GOLD_DIR}\dim_players")
print("Status               : SUCCESS")
print("=" * 80)


In [ ]:
# ==============================================================================
# Dataset : dim_teams
# Layer   : Silver -> Gold
# Purpose : Build Star Schema (Join with Dimensions)
# ==============================================================================

print("=" * 80)
print("Processing Gold Table : dim_teams")
print("=" * 80)


In [ ]:
# ------------------------------------------------------------------------------
# Step 1 : Read Silver Dataset
# ------------------------------------------------------------------------------
dataset = "teams_ids"
gold_name = "dim_teams"
folder = "ids"

df = spark.read.parquet(os.path.join(SILVER_DIR, folder, dataset))
rows_before = df.count()
print(f"Silver Rows Read : {rows_before}")
df.show(5)


In [ ]:
# ------------------------------------------------------------------------------
# Step 3 : Write Gold Dataset
# ------------------------------------------------------------------------------
df.coalesce(1).write.mode("overwrite").csv(os.path.join(GOLD_DIR, gold_name), header=True)


In [ ]:
# ------------------------------------------------------------------------------
# Step 4 : ETL Summary
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("ETL EXECUTION SUMMARY")
print("=" * 80)
print(f"Gold Table           : dim_teams")
print(f"Rows Written         : {df.count()}")
print(f"Gold Location        : {GOLD_DIR}\dim_teams")
print("Status               : SUCCESS")
print("=" * 80)


In [ ]:
# ==============================================================================
# Dataset : dim_match_types
# Layer   : Silver -> Gold
# Purpose : Build Star Schema (Join with Dimensions)
# ==============================================================================

print("=" * 80)
print("Processing Gold Table : dim_match_types")
print("=" * 80)


In [ ]:
# ------------------------------------------------------------------------------
# Step 1 : Read Silver Dataset
# ------------------------------------------------------------------------------
dataset = "tournaments_stages_match_types_ids"
gold_name = "dim_match_types"
folder = "ids"

df = spark.read.parquet(os.path.join(SILVER_DIR, folder, dataset))
rows_before = df.count()
print(f"Silver Rows Read : {rows_before}")
df.show(5)


In [ ]:
# ------------------------------------------------------------------------------
# Step 3 : Write Gold Dataset
# ------------------------------------------------------------------------------
df.coalesce(1).write.mode("overwrite").csv(os.path.join(GOLD_DIR, gold_name), header=True)


In [ ]:
# ------------------------------------------------------------------------------
# Step 4 : ETL Summary
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("ETL EXECUTION SUMMARY")
print("=" * 80)
print(f"Gold Table           : dim_match_types")
print(f"Rows Written         : {df.count()}")
print(f"Gold Location        : {GOLD_DIR}\dim_match_types")
print("Status               : SUCCESS")
print("=" * 80)


In [ ]:
# ==============================================================================
# Dataset : dim_games
# Layer   : Silver -> Gold
# Purpose : Build Star Schema (Join with Dimensions)
# ==============================================================================

print("=" * 80)
print("Processing Gold Table : dim_games")
print("=" * 80)


In [ ]:
# ------------------------------------------------------------------------------
# Step 1 : Read Silver Dataset
# ------------------------------------------------------------------------------
dataset = "tournaments_stages_matches_games_ids"
gold_name = "dim_games"
folder = "ids"

df = spark.read.parquet(os.path.join(SILVER_DIR, folder, dataset))
rows_before = df.count()
print(f"Silver Rows Read : {rows_before}")
df.show(5)


In [ ]:
# ------------------------------------------------------------------------------
# Step 3 : Write Gold Dataset
# ------------------------------------------------------------------------------
df.coalesce(1).write.mode("overwrite").csv(os.path.join(GOLD_DIR, gold_name), header=True)


In [ ]:
# ------------------------------------------------------------------------------
# Step 4 : ETL Summary
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("ETL EXECUTION SUMMARY")
print("=" * 80)
print(f"Gold Table           : dim_games")
print(f"Rows Written         : {df.count()}")
print(f"Gold Location        : {GOLD_DIR}\dim_games")
print("Status               : SUCCESS")
print("=" * 80)


In [ ]:
# ==============================================================================
# Dataset : fact_agents_pick_rates
# Layer   : Silver -> Gold
# Purpose : Build Star Schema (Join with Dimensions)
# ==============================================================================

print("=" * 80)
print("Processing Gold Table : fact_agents_pick_rates")
print("=" * 80)


In [ ]:
# ------------------------------------------------------------------------------
# Step 1 : Read Silver Dataset
# ------------------------------------------------------------------------------
dataset = "agents_pick_rates"
gold_name = "fact_agents_pick_rates"
folder = "agents"

df = spark.read.parquet(os.path.join(SILVER_DIR, folder, dataset))
rows_before = df.count()
print(f"Silver Rows Read : {rows_before}")
df.show(5)


In [ ]:
# ------------------------------------------------------------------------------
# Step 2 : Join with Dimensions
# ------------------------------------------------------------------------------
columns_lower = {c.lower(): c for c in df.columns}
if 'player' in columns_lower:
    p_col = columns_lower['player']
    df = df.join(players_ids, df[p_col] == players_ids["join_player"], "left").drop(p_col, "join_player")
if 'team' in columns_lower:
    t_col = columns_lower['team']
    df = df.join(teams_ids, df[t_col] == teams_ids["join_team"], "left").drop(t_col, "join_team")


In [ ]:
# ------------------------------------------------------------------------------
# Step 3 : Write Gold Dataset
# ------------------------------------------------------------------------------
df.coalesce(1).write.mode("overwrite").csv(os.path.join(GOLD_DIR, gold_name), header=True)


In [ ]:
# ------------------------------------------------------------------------------
# Step 4 : ETL Summary
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("ETL EXECUTION SUMMARY")
print("=" * 80)
print(f"Gold Table           : fact_agents_pick_rates")
print(f"Rows Written         : {df.count()}")
print(f"Gold Location        : {GOLD_DIR}\fact_agents_pick_rates")
print("Status               : SUCCESS")
print("=" * 80)


In [ ]:
# ==============================================================================
# Dataset : fact_maps_stats_agents
# Layer   : Silver -> Gold
# Purpose : Build Star Schema (Join with Dimensions)
# ==============================================================================

print("=" * 80)
print("Processing Gold Table : fact_maps_stats_agents")
print("=" * 80)


In [ ]:
# ------------------------------------------------------------------------------
# Step 1 : Read Silver Dataset
# ------------------------------------------------------------------------------
dataset = "maps_stats"
gold_name = "fact_maps_stats_agents"
folder = "agents"

df = spark.read.parquet(os.path.join(SILVER_DIR, folder, dataset))
rows_before = df.count()
print(f"Silver Rows Read : {rows_before}")
df.show(5)


In [ ]:
# ------------------------------------------------------------------------------
# Step 2 : Join with Dimensions
# ------------------------------------------------------------------------------
columns_lower = {c.lower(): c for c in df.columns}
if 'player' in columns_lower:
    p_col = columns_lower['player']
    df = df.join(players_ids, df[p_col] == players_ids["join_player"], "left").drop(p_col, "join_player")
if 'team' in columns_lower:
    t_col = columns_lower['team']
    df = df.join(teams_ids, df[t_col] == teams_ids["join_team"], "left").drop(t_col, "join_team")


In [ ]:
# ------------------------------------------------------------------------------
# Step 3 : Write Gold Dataset
# ------------------------------------------------------------------------------
df.coalesce(1).write.mode("overwrite").csv(os.path.join(GOLD_DIR, gold_name), header=True)


In [ ]:
# ------------------------------------------------------------------------------
# Step 4 : ETL Summary
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("ETL EXECUTION SUMMARY")
print("=" * 80)
print(f"Gold Table           : fact_maps_stats_agents")
print(f"Rows Written         : {df.count()}")
print(f"Gold Location        : {GOLD_DIR}\fact_maps_stats_agents")
print("Status               : SUCCESS")
print("=" * 80)


In [ ]:
# ==============================================================================
# Dataset : fact_teams_picked_agents
# Layer   : Silver -> Gold
# Purpose : Build Star Schema (Join with Dimensions)
# ==============================================================================

print("=" * 80)
print("Processing Gold Table : fact_teams_picked_agents")
print("=" * 80)


In [ ]:
# ------------------------------------------------------------------------------
# Step 1 : Read Silver Dataset
# ------------------------------------------------------------------------------
dataset = "teams_picked_agents"
gold_name = "fact_teams_picked_agents"
folder = "agents"

df = spark.read.parquet(os.path.join(SILVER_DIR, folder, dataset))
rows_before = df.count()
print(f"Silver Rows Read : {rows_before}")
df.show(5)


In [ ]:
# ------------------------------------------------------------------------------
# Step 2 : Join with Dimensions
# ------------------------------------------------------------------------------
columns_lower = {c.lower(): c for c in df.columns}
if 'player' in columns_lower:
    p_col = columns_lower['player']
    df = df.join(players_ids, df[p_col] == players_ids["join_player"], "left").drop(p_col, "join_player")
if 'team' in columns_lower:
    t_col = columns_lower['team']
    df = df.join(teams_ids, df[t_col] == teams_ids["join_team"], "left").drop(t_col, "join_team")


In [ ]:
# ------------------------------------------------------------------------------
# Step 3 : Write Gold Dataset
# ------------------------------------------------------------------------------
df.coalesce(1).write.mode("overwrite").csv(os.path.join(GOLD_DIR, gold_name), header=True)


In [ ]:
# ------------------------------------------------------------------------------
# Step 4 : ETL Summary
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("ETL EXECUTION SUMMARY")
print("=" * 80)
print(f"Gold Table           : fact_teams_picked_agents")
print(f"Rows Written         : {df.count()}")
print(f"Gold Location        : {GOLD_DIR}\fact_teams_picked_agents")
print("Status               : SUCCESS")
print("=" * 80)


In [ ]:
# ==============================================================================
# Dataset : fact_player_stats
# Layer   : Silver -> Gold
# Purpose : Build Star Schema (Join with Dimensions)
# ==============================================================================

print("=" * 80)
print("Processing Gold Table : fact_player_stats")
print("=" * 80)


In [ ]:
# ------------------------------------------------------------------------------
# Step 1 : Read Silver Dataset
# ------------------------------------------------------------------------------
dataset = "players_stats"
gold_name = "fact_player_stats"
folder = "players_stats"

df = spark.read.parquet(os.path.join(SILVER_DIR, folder, dataset))
rows_before = df.count()
print(f"Silver Rows Read : {rows_before}")
df.show(5)


In [ ]:
# ------------------------------------------------------------------------------
# Step 2 : Join with Dimensions
# ------------------------------------------------------------------------------
columns_lower = {c.lower(): c for c in df.columns}
if 'player' in columns_lower:
    p_col = columns_lower['player']
    df = df.join(players_ids, df[p_col] == players_ids["join_player"], "left").drop(p_col, "join_player")
if 'team' in columns_lower:
    t_col = columns_lower['team']
    df = df.join(teams_ids, df[t_col] == teams_ids["join_team"], "left").drop(t_col, "join_team")


In [ ]:
# ------------------------------------------------------------------------------
# Step 3 : Write Gold Dataset
# ------------------------------------------------------------------------------
df.coalesce(1).write.mode("overwrite").csv(os.path.join(GOLD_DIR, gold_name), header=True)


In [ ]:
# ------------------------------------------------------------------------------
# Step 4 : ETL Summary
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("ETL EXECUTION SUMMARY")
print("=" * 80)
print(f"Gold Table           : fact_player_stats")
print(f"Rows Written         : {df.count()}")
print(f"Gold Location        : {GOLD_DIR}\fact_player_stats")
print("Status               : SUCCESS")
print("=" * 80)


In [ ]:
# ==============================================================================
# Dataset : fact_draft_phase
# Layer   : Silver -> Gold
# Purpose : Build Star Schema (Join with Dimensions)
# ==============================================================================

print("=" * 80)
print("Processing Gold Table : fact_draft_phase")
print("=" * 80)


In [ ]:
# ------------------------------------------------------------------------------
# Step 1 : Read Silver Dataset
# ------------------------------------------------------------------------------
dataset = "draft_phase"
gold_name = "fact_draft_phase"
folder = "matches"

df = spark.read.parquet(os.path.join(SILVER_DIR, folder, dataset))
rows_before = df.count()
print(f"Silver Rows Read : {rows_before}")
df.show(5)


In [ ]:
# ------------------------------------------------------------------------------
# Step 2 : Join with Dimensions
# ------------------------------------------------------------------------------
columns_lower = {c.lower(): c for c in df.columns}
if 'player' in columns_lower:
    p_col = columns_lower['player']
    df = df.join(players_ids, df[p_col] == players_ids["join_player"], "left").drop(p_col, "join_player")
if 'team' in columns_lower:
    t_col = columns_lower['team']
    df = df.join(teams_ids, df[t_col] == teams_ids["join_team"], "left").drop(t_col, "join_team")


In [ ]:
# ------------------------------------------------------------------------------
# Step 3 : Write Gold Dataset
# ------------------------------------------------------------------------------
df.coalesce(1).write.mode("overwrite").csv(os.path.join(GOLD_DIR, gold_name), header=True)


In [ ]:
# ------------------------------------------------------------------------------
# Step 4 : ETL Summary
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("ETL EXECUTION SUMMARY")
print("=" * 80)
print(f"Gold Table           : fact_draft_phase")
print(f"Rows Written         : {df.count()}")
print(f"Gold Location        : {GOLD_DIR}\fact_draft_phase")
print("Status               : SUCCESS")
print("=" * 80)


In [ ]:
# ==============================================================================
# Dataset : fact_eco_rounds
# Layer   : Silver -> Gold
# Purpose : Build Star Schema (Join with Dimensions)
# ==============================================================================

print("=" * 80)
print("Processing Gold Table : fact_eco_rounds")
print("=" * 80)


In [ ]:
# ------------------------------------------------------------------------------
# Step 1 : Read Silver Dataset
# ------------------------------------------------------------------------------
dataset = "eco_rounds"
gold_name = "fact_eco_rounds"
folder = "matches"

df = spark.read.parquet(os.path.join(SILVER_DIR, folder, dataset))
rows_before = df.count()
print(f"Silver Rows Read : {rows_before}")
df.show(5)


In [ ]:
# ------------------------------------------------------------------------------
# Step 2 : Join with Dimensions
# ------------------------------------------------------------------------------
columns_lower = {c.lower(): c for c in df.columns}
if 'player' in columns_lower:
    p_col = columns_lower['player']
    df = df.join(players_ids, df[p_col] == players_ids["join_player"], "left").drop(p_col, "join_player")
if 'team' in columns_lower:
    t_col = columns_lower['team']
    df = df.join(teams_ids, df[t_col] == teams_ids["join_team"], "left").drop(t_col, "join_team")


In [ ]:
# ------------------------------------------------------------------------------
# Step 3 : Write Gold Dataset
# ------------------------------------------------------------------------------
df.coalesce(1).write.mode("overwrite").csv(os.path.join(GOLD_DIR, gold_name), header=True)


In [ ]:
# ------------------------------------------------------------------------------
# Step 4 : ETL Summary
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("ETL EXECUTION SUMMARY")
print("=" * 80)
print(f"Gold Table           : fact_eco_rounds")
print(f"Rows Written         : {df.count()}")
print(f"Gold Location        : {GOLD_DIR}\fact_eco_rounds")
print("Status               : SUCCESS")
print("=" * 80)


In [ ]:
# ==============================================================================
# Dataset : fact_eco_stats
# Layer   : Silver -> Gold
# Purpose : Build Star Schema (Join with Dimensions)
# ==============================================================================

print("=" * 80)
print("Processing Gold Table : fact_eco_stats")
print("=" * 80)


In [ ]:
# ------------------------------------------------------------------------------
# Step 1 : Read Silver Dataset
# ------------------------------------------------------------------------------
dataset = "eco_stats"
gold_name = "fact_eco_stats"
folder = "matches"

df = spark.read.parquet(os.path.join(SILVER_DIR, folder, dataset))
rows_before = df.count()
print(f"Silver Rows Read : {rows_before}")
df.show(5)


In [ ]:
# ------------------------------------------------------------------------------
# Step 2 : Join with Dimensions
# ------------------------------------------------------------------------------
columns_lower = {c.lower(): c for c in df.columns}
if 'player' in columns_lower:
    p_col = columns_lower['player']
    df = df.join(players_ids, df[p_col] == players_ids["join_player"], "left").drop(p_col, "join_player")
if 'team' in columns_lower:
    t_col = columns_lower['team']
    df = df.join(teams_ids, df[t_col] == teams_ids["join_team"], "left").drop(t_col, "join_team")


In [ ]:
# ------------------------------------------------------------------------------
# Step 3 : Write Gold Dataset
# ------------------------------------------------------------------------------
df.coalesce(1).write.mode("overwrite").csv(os.path.join(GOLD_DIR, gold_name), header=True)


In [ ]:
# ------------------------------------------------------------------------------
# Step 4 : ETL Summary
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("ETL EXECUTION SUMMARY")
print("=" * 80)
print(f"Gold Table           : fact_eco_stats")
print(f"Rows Written         : {df.count()}")
print(f"Gold Location        : {GOLD_DIR}\fact_eco_stats")
print("Status               : SUCCESS")
print("=" * 80)


In [ ]:
# ==============================================================================
# Dataset : fact_kills
# Layer   : Silver -> Gold
# Purpose : Build Star Schema (Join with Dimensions)
# ==============================================================================

print("=" * 80)
print("Processing Gold Table : fact_kills")
print("=" * 80)


In [ ]:
# ------------------------------------------------------------------------------
# Step 1 : Read Silver Dataset
# ------------------------------------------------------------------------------
dataset = "kills"
gold_name = "fact_kills"
folder = "matches"

df = spark.read.parquet(os.path.join(SILVER_DIR, folder, dataset))
rows_before = df.count()
print(f"Silver Rows Read : {rows_before}")
df.show(5)


In [ ]:
# ------------------------------------------------------------------------------
# Step 2 : Join with Dimensions
# ------------------------------------------------------------------------------
columns_lower = {c.lower(): c for c in df.columns}
if 'player' in columns_lower:
    p_col = columns_lower['player']
    df = df.join(players_ids, df[p_col] == players_ids["join_player"], "left").drop(p_col, "join_player")
if 'team' in columns_lower:
    t_col = columns_lower['team']
    df = df.join(teams_ids, df[t_col] == teams_ids["join_team"], "left").drop(t_col, "join_team")


In [ ]:
# ------------------------------------------------------------------------------
# Step 3 : Write Gold Dataset
# ------------------------------------------------------------------------------
df.coalesce(1).write.mode("overwrite").csv(os.path.join(GOLD_DIR, gold_name), header=True)


In [ ]:
# ------------------------------------------------------------------------------
# Step 4 : ETL Summary
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("ETL EXECUTION SUMMARY")
print("=" * 80)
print(f"Gold Table           : fact_kills")
print(f"Rows Written         : {df.count()}")
print(f"Gold Location        : {GOLD_DIR}\fact_kills")
print("Status               : SUCCESS")
print("=" * 80)


In [ ]:
# ==============================================================================
# Dataset : fact_kills_stats
# Layer   : Silver -> Gold
# Purpose : Build Star Schema (Join with Dimensions)
# ==============================================================================

print("=" * 80)
print("Processing Gold Table : fact_kills_stats")
print("=" * 80)


In [ ]:
# ------------------------------------------------------------------------------
# Step 1 : Read Silver Dataset
# ------------------------------------------------------------------------------
dataset = "kills_stats"
gold_name = "fact_kills_stats"
folder = "matches"

df = spark.read.parquet(os.path.join(SILVER_DIR, folder, dataset))
rows_before = df.count()
print(f"Silver Rows Read : {rows_before}")
df.show(5)


In [ ]:
# ------------------------------------------------------------------------------
# Step 2 : Join with Dimensions
# ------------------------------------------------------------------------------
columns_lower = {c.lower(): c for c in df.columns}
if 'player' in columns_lower:
    p_col = columns_lower['player']
    df = df.join(players_ids, df[p_col] == players_ids["join_player"], "left").drop(p_col, "join_player")
if 'team' in columns_lower:
    t_col = columns_lower['team']
    df = df.join(teams_ids, df[t_col] == teams_ids["join_team"], "left").drop(t_col, "join_team")


In [ ]:
# ------------------------------------------------------------------------------
# Step 3 : Write Gold Dataset
# ------------------------------------------------------------------------------
df.coalesce(1).write.mode("overwrite").csv(os.path.join(GOLD_DIR, gold_name), header=True)


In [ ]:
# ------------------------------------------------------------------------------
# Step 4 : ETL Summary
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("ETL EXECUTION SUMMARY")
print("=" * 80)
print(f"Gold Table           : fact_kills_stats")
print(f"Rows Written         : {df.count()}")
print(f"Gold Location        : {GOLD_DIR}\fact_kills_stats")
print("Status               : SUCCESS")
print("=" * 80)


In [ ]:
# ==============================================================================
# Dataset : fact_maps_played
# Layer   : Silver -> Gold
# Purpose : Build Star Schema (Join with Dimensions)
# ==============================================================================

print("=" * 80)
print("Processing Gold Table : fact_maps_played")
print("=" * 80)


In [ ]:
# ------------------------------------------------------------------------------
# Step 1 : Read Silver Dataset
# ------------------------------------------------------------------------------
dataset = "maps_played"
gold_name = "fact_maps_played"
folder = "matches"

df = spark.read.parquet(os.path.join(SILVER_DIR, folder, dataset))
rows_before = df.count()
print(f"Silver Rows Read : {rows_before}")
df.show(5)


In [ ]:
# ------------------------------------------------------------------------------
# Step 2 : Join with Dimensions
# ------------------------------------------------------------------------------
columns_lower = {c.lower(): c for c in df.columns}
if 'player' in columns_lower:
    p_col = columns_lower['player']
    df = df.join(players_ids, df[p_col] == players_ids["join_player"], "left").drop(p_col, "join_player")
if 'team' in columns_lower:
    t_col = columns_lower['team']
    df = df.join(teams_ids, df[t_col] == teams_ids["join_team"], "left").drop(t_col, "join_team")


In [ ]:
# ------------------------------------------------------------------------------
# Step 3 : Write Gold Dataset
# ------------------------------------------------------------------------------
df.coalesce(1).write.mode("overwrite").csv(os.path.join(GOLD_DIR, gold_name), header=True)


In [ ]:
# ------------------------------------------------------------------------------
# Step 4 : ETL Summary
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("ETL EXECUTION SUMMARY")
print("=" * 80)
print(f"Gold Table           : fact_maps_played")
print(f"Rows Written         : {df.count()}")
print(f"Gold Location        : {GOLD_DIR}\fact_maps_played")
print("Status               : SUCCESS")
print("=" * 80)


In [ ]:
# ==============================================================================
# Dataset : fact_maps_scores
# Layer   : Silver -> Gold
# Purpose : Build Star Schema (Join with Dimensions)
# ==============================================================================

print("=" * 80)
print("Processing Gold Table : fact_maps_scores")
print("=" * 80)


In [ ]:
# ------------------------------------------------------------------------------
# Step 1 : Read Silver Dataset
# ------------------------------------------------------------------------------
dataset = "maps_scores"
gold_name = "fact_maps_scores"
folder = "matches"

df = spark.read.parquet(os.path.join(SILVER_DIR, folder, dataset))
rows_before = df.count()
print(f"Silver Rows Read : {rows_before}")
df.show(5)


In [ ]:
# ------------------------------------------------------------------------------
# Step 2 : Join with Dimensions
# ------------------------------------------------------------------------------
columns_lower = {c.lower(): c for c in df.columns}
if 'player' in columns_lower:
    p_col = columns_lower['player']
    df = df.join(players_ids, df[p_col] == players_ids["join_player"], "left").drop(p_col, "join_player")
if 'team' in columns_lower:
    t_col = columns_lower['team']
    df = df.join(teams_ids, df[t_col] == teams_ids["join_team"], "left").drop(t_col, "join_team")


In [ ]:
# ------------------------------------------------------------------------------
# Step 3 : Write Gold Dataset
# ------------------------------------------------------------------------------
df.coalesce(1).write.mode("overwrite").csv(os.path.join(GOLD_DIR, gold_name), header=True)


In [ ]:
# ------------------------------------------------------------------------------
# Step 4 : ETL Summary
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("ETL EXECUTION SUMMARY")
print("=" * 80)
print(f"Gold Table           : fact_maps_scores")
print(f"Rows Written         : {df.count()}")
print(f"Gold Location        : {GOLD_DIR}\fact_maps_scores")
print("Status               : SUCCESS")
print("=" * 80)


In [ ]:
# ==============================================================================
# Dataset : fact_matches_overview
# Layer   : Silver -> Gold
# Purpose : Build Star Schema (Join with Dimensions)
# ==============================================================================

print("=" * 80)
print("Processing Gold Table : fact_matches_overview")
print("=" * 80)


In [ ]:
# ------------------------------------------------------------------------------
# Step 1 : Read Silver Dataset
# ------------------------------------------------------------------------------
dataset = "overview"
gold_name = "fact_matches_overview"
folder = "matches"

df = spark.read.parquet(os.path.join(SILVER_DIR, folder, dataset))
rows_before = df.count()
print(f"Silver Rows Read : {rows_before}")
df.show(5)


In [ ]:
# ------------------------------------------------------------------------------
# Step 2 : Join with Dimensions
# ------------------------------------------------------------------------------
columns_lower = {c.lower(): c for c in df.columns}
if 'player' in columns_lower:
    p_col = columns_lower['player']
    df = df.join(players_ids, df[p_col] == players_ids["join_player"], "left").drop(p_col, "join_player")
if 'team' in columns_lower:
    t_col = columns_lower['team']
    df = df.join(teams_ids, df[t_col] == teams_ids["join_team"], "left").drop(t_col, "join_team")


In [ ]:
# ------------------------------------------------------------------------------
# Step 3 : Write Gold Dataset
# ------------------------------------------------------------------------------
df.coalesce(1).write.mode("overwrite").csv(os.path.join(GOLD_DIR, gold_name), header=True)


In [ ]:
# ------------------------------------------------------------------------------
# Step 4 : ETL Summary
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("ETL EXECUTION SUMMARY")
print("=" * 80)
print(f"Gold Table           : fact_matches_overview")
print(f"Rows Written         : {df.count()}")
print(f"Gold Location        : {GOLD_DIR}\fact_matches_overview")
print("Status               : SUCCESS")
print("=" * 80)


In [ ]:
# ==============================================================================
# Dataset : fact_rounds_kills
# Layer   : Silver -> Gold
# Purpose : Build Star Schema (Join with Dimensions)
# ==============================================================================

print("=" * 80)
print("Processing Gold Table : fact_rounds_kills")
print("=" * 80)


In [ ]:
# ------------------------------------------------------------------------------
# Step 1 : Read Silver Dataset
# ------------------------------------------------------------------------------
dataset = "rounds_kills"
gold_name = "fact_rounds_kills"
folder = "matches"

df = spark.read.parquet(os.path.join(SILVER_DIR, folder, dataset))
rows_before = df.count()
print(f"Silver Rows Read : {rows_before}")
df.show(5)


In [ ]:
# ------------------------------------------------------------------------------
# Step 2 : Join with Dimensions
# ------------------------------------------------------------------------------
columns_lower = {c.lower(): c for c in df.columns}
if 'player' in columns_lower:
    p_col = columns_lower['player']
    df = df.join(players_ids, df[p_col] == players_ids["join_player"], "left").drop(p_col, "join_player")
if 'team' in columns_lower:
    t_col = columns_lower['team']
    df = df.join(teams_ids, df[t_col] == teams_ids["join_team"], "left").drop(t_col, "join_team")


In [ ]:
# ------------------------------------------------------------------------------
# Step 3 : Write Gold Dataset
# ------------------------------------------------------------------------------
df.coalesce(1).write.mode("overwrite").csv(os.path.join(GOLD_DIR, gold_name), header=True)


In [ ]:
# ------------------------------------------------------------------------------
# Step 4 : ETL Summary
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("ETL EXECUTION SUMMARY")
print("=" * 80)
print(f"Gold Table           : fact_rounds_kills")
print(f"Rows Written         : {df.count()}")
print(f"Gold Location        : {GOLD_DIR}\fact_rounds_kills")
print("Status               : SUCCESS")
print("=" * 80)


In [ ]:
# ==============================================================================
# Dataset : fact_scores
# Layer   : Silver -> Gold
# Purpose : Build Star Schema (Join with Dimensions)
# ==============================================================================

print("=" * 80)
print("Processing Gold Table : fact_scores")
print("=" * 80)


In [ ]:
# ------------------------------------------------------------------------------
# Step 1 : Read Silver Dataset
# ------------------------------------------------------------------------------
dataset = "scores"
gold_name = "fact_scores"
folder = "matches"

df = spark.read.parquet(os.path.join(SILVER_DIR, folder, dataset))
rows_before = df.count()
print(f"Silver Rows Read : {rows_before}")
df.show(5)


In [ ]:
# ------------------------------------------------------------------------------
# Step 2 : Join with Dimensions
# ------------------------------------------------------------------------------
columns_lower = {c.lower(): c for c in df.columns}
if 'player' in columns_lower:
    p_col = columns_lower['player']
    df = df.join(players_ids, df[p_col] == players_ids["join_player"], "left").drop(p_col, "join_player")
if 'team' in columns_lower:
    t_col = columns_lower['team']
    df = df.join(teams_ids, df[t_col] == teams_ids["join_team"], "left").drop(t_col, "join_team")


In [ ]:
# ------------------------------------------------------------------------------
# Step 3 : Write Gold Dataset
# ------------------------------------------------------------------------------
df.coalesce(1).write.mode("overwrite").csv(os.path.join(GOLD_DIR, gold_name), header=True)


In [ ]:
# ------------------------------------------------------------------------------
# Step 4 : ETL Summary
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("ETL EXECUTION SUMMARY")
print("=" * 80)
print(f"Gold Table           : fact_scores")
print(f"Rows Written         : {df.count()}")
print(f"Gold Location        : {GOLD_DIR}\fact_scores")
print("Status               : SUCCESS")
print("=" * 80)


In [ ]:
# ==============================================================================
# Dataset : fact_team_mapping
# Layer   : Silver -> Gold
# Purpose : Build Star Schema (Join with Dimensions)
# ==============================================================================

print("=" * 80)
print("Processing Gold Table : fact_team_mapping")
print("=" * 80)


In [ ]:
# ------------------------------------------------------------------------------
# Step 1 : Read Silver Dataset
# ------------------------------------------------------------------------------
dataset = "team_mapping"
gold_name = "fact_team_mapping"
folder = "matches"

df = spark.read.parquet(os.path.join(SILVER_DIR, folder, dataset))
rows_before = df.count()
print(f"Silver Rows Read : {rows_before}")
df.show(5)


In [ ]:
# ------------------------------------------------------------------------------
# Step 2 : Join with Dimensions
# ------------------------------------------------------------------------------
columns_lower = {c.lower(): c for c in df.columns}
if 'player' in columns_lower:
    p_col = columns_lower['player']
    df = df.join(players_ids, df[p_col] == players_ids["join_player"], "left").drop(p_col, "join_player")
if 'team' in columns_lower:
    t_col = columns_lower['team']
    df = df.join(teams_ids, df[t_col] == teams_ids["join_team"], "left").drop(t_col, "join_team")


In [ ]:
# ------------------------------------------------------------------------------
# Step 3 : Write Gold Dataset
# ------------------------------------------------------------------------------
df.coalesce(1).write.mode("overwrite").csv(os.path.join(GOLD_DIR, gold_name), header=True)


In [ ]:
# ------------------------------------------------------------------------------
# Step 4 : ETL Summary
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("ETL EXECUTION SUMMARY")
print("=" * 80)
print(f"Gold Table           : fact_team_mapping")
print(f"Rows Written         : {df.count()}")
print(f"Gold Location        : {GOLD_DIR}\fact_team_mapping")
print("Status               : SUCCESS")
print("=" * 80)


In [ ]:
# ==============================================================================
# Dataset : fact_win_loss_methods_count
# Layer   : Silver -> Gold
# Purpose : Build Star Schema (Join with Dimensions)
# ==============================================================================

print("=" * 80)
print("Processing Gold Table : fact_win_loss_methods_count")
print("=" * 80)


In [ ]:
# ------------------------------------------------------------------------------
# Step 1 : Read Silver Dataset
# ------------------------------------------------------------------------------
dataset = "win_loss_methods_count"
gold_name = "fact_win_loss_methods_count"
folder = "matches"

df = spark.read.parquet(os.path.join(SILVER_DIR, folder, dataset))
rows_before = df.count()
print(f"Silver Rows Read : {rows_before}")
df.show(5)


In [ ]:
# ------------------------------------------------------------------------------
# Step 2 : Join with Dimensions
# ------------------------------------------------------------------------------
columns_lower = {c.lower(): c for c in df.columns}
if 'player' in columns_lower:
    p_col = columns_lower['player']
    df = df.join(players_ids, df[p_col] == players_ids["join_player"], "left").drop(p_col, "join_player")
if 'team' in columns_lower:
    t_col = columns_lower['team']
    df = df.join(teams_ids, df[t_col] == teams_ids["join_team"], "left").drop(t_col, "join_team")


In [ ]:
# ------------------------------------------------------------------------------
# Step 3 : Write Gold Dataset
# ------------------------------------------------------------------------------
df.coalesce(1).write.mode("overwrite").csv(os.path.join(GOLD_DIR, gold_name), header=True)


In [ ]:
# ------------------------------------------------------------------------------
# Step 4 : ETL Summary
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("ETL EXECUTION SUMMARY")
print("=" * 80)
print(f"Gold Table           : fact_win_loss_methods_count")
print(f"Rows Written         : {df.count()}")
print(f"Gold Location        : {GOLD_DIR}\fact_win_loss_methods_count")
print("Status               : SUCCESS")
print("=" * 80)


In [ ]:
# ==============================================================================
# Dataset : fact_win_loss_methods_round_number
# Layer   : Silver -> Gold
# Purpose : Build Star Schema (Join with Dimensions)
# ==============================================================================

print("=" * 80)
print("Processing Gold Table : fact_win_loss_methods_round_number")
print("=" * 80)


In [ ]:
# ------------------------------------------------------------------------------
# Step 1 : Read Silver Dataset
# ------------------------------------------------------------------------------
dataset = "win_loss_methods_round_number"
gold_name = "fact_win_loss_methods_round_number"
folder = "matches"

df = spark.read.parquet(os.path.join(SILVER_DIR, folder, dataset))
rows_before = df.count()
print(f"Silver Rows Read : {rows_before}")
df.show(5)


In [ ]:
# ------------------------------------------------------------------------------
# Step 2 : Join with Dimensions
# ------------------------------------------------------------------------------
columns_lower = {c.lower(): c for c in df.columns}
if 'player' in columns_lower:
    p_col = columns_lower['player']
    df = df.join(players_ids, df[p_col] == players_ids["join_player"], "left").drop(p_col, "join_player")
if 'team' in columns_lower:
    t_col = columns_lower['team']
    df = df.join(teams_ids, df[t_col] == teams_ids["join_team"], "left").drop(t_col, "join_team")


In [ ]:
# ------------------------------------------------------------------------------
# Step 3 : Write Gold Dataset
# ------------------------------------------------------------------------------
df.coalesce(1).write.mode("overwrite").csv(os.path.join(GOLD_DIR, gold_name), header=True)


In [ ]:
# ------------------------------------------------------------------------------
# Step 4 : ETL Summary
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("ETL EXECUTION SUMMARY")
print("=" * 80)
print(f"Gold Table           : fact_win_loss_methods_round_number")
print(f"Rows Written         : {df.count()}")
print(f"Gold Location        : {GOLD_DIR}\fact_win_loss_methods_round_number")
print("Status               : SUCCESS")
print("=" * 80)
